# 04.03 - Compare Two GraphDefinitions (definition ↔ definition)

`compare_definitions(left, right)` runs a **symmetric diff** between two
`GraphDefinition` objects. It answers:

> *What changed between the old and new declared graph schema?*

Use it to detect schema drift between versions of your domain model, or
to compare two variant schemas (e.g. v1 vs v2).

All issues are `Severity.INFO`; the result is always `is_valid = True`.

This notebook:
1. Builds two `GraphDefinition` objects with deliberate differences.
2. Calls `compare_definitions(old, new)` and displays the diff.
3. Shows the `PROPERTY_TYPE_CHANGED` and `CARDINALITY_CHANGED` cases.

In [ ]:
from typing import Optional

from orthograph.comparison.engine import compare_definitions
from orthograph.graph_definition.graph_definition import GraphDefinition
from orthograph.graph_definition.models import (
    Cardinality,
    NodeModel,
    RelationshipModel,
)

## 1. Define the 'old' schema (v1)

The v1 filmography model: Person, Movie, City and the ACTED_IN / LIVES_IN
relationship types.

In [ ]:
# --- v1 node types ---


class PersonV1(NodeModel):
    __label__ = "Person"
    name: str
    born: Optional[int] = None


class MovieV1(NodeModel):
    __label__ = "Movie"
    title: str
    released: int


class CityV1(NodeModel):
    __label__ = "City"
    name: str


# --- v1 relationship types ---


class ActedInV1(RelationshipModel):
    __label__ = "ACTED_IN"
    __source_label__ = "Person"
    __target_label__ = "Movie"
    role: str


class LivesInV1(RelationshipModel):
    __label__ = "LIVES_IN"
    __source_label__ = "Person"
    __target_label__ = "City"
    __source_cardinality__ = Cardinality.ONE


schema_v1 = GraphDefinition(
    name="filmography_v1",
    node_types=[PersonV1, MovieV1, CityV1],
    relationship_types=[ActedInV1, LivesInV1],
)

print("v1 nodes:", sorted(schema_v1.node_labels))
print("v1 rels :", sorted(schema_v1.relationship_labels))

## 2. Define the 'new' schema (v2)

Changes introduced in v2:
- `City` node removed; `born` property removed from `Person`.
- `Person.name` type changed from `str` to `int` (intentional demo; bad idea in practice!).
- New `Genre` node and `IN_GENRE` relationship.
- `LIVES_IN` cardinality relaxed from `ONE` to `ZERO_OR_MORE`.
- `ACTED_IN` now points at `Movie` from both `Person` and `Director` sides
  (endpoints change).

In [ ]:
# --- v2 node types ---


class PersonV2(NodeModel):
    __label__ = "Person"
    name: int  # changed type!
    email: Optional[str] = None  # new property


class MovieV2(NodeModel):
    __label__ = "Movie"
    title: str
    released: int


class GenreV2(NodeModel):
    __label__ = "Genre"
    name: str


class DirectorV2(NodeModel):
    __label__ = "Director"
    name: str


# --- v2 relationship types ---


class ActedInV2(RelationshipModel):
    __label__ = "ACTED_IN"
    __source_label__ = "Director"  # endpoint changed!
    __target_label__ = "Movie"
    role: str


class LivesInV2(RelationshipModel):
    __label__ = "LIVES_IN"
    __source_label__ = "Person"
    __target_label__ = "Genre"  # target changed!
    __source_cardinality__ = Cardinality.ZERO_OR_MORE  # relaxed


class InGenreV2(RelationshipModel):
    __label__ = "IN_GENRE"
    __source_label__ = "Movie"
    __target_label__ = "Genre"


schema_v2 = GraphDefinition(
    name="filmography_v2",
    node_types=[PersonV2, MovieV2, GenreV2, DirectorV2],
    relationship_types=[ActedInV2, LivesInV2, InGenreV2],
)

print("v2 nodes:", sorted(schema_v2.node_labels))
print("v2 rels :", sorted(schema_v2.relationship_labels))

## 3. Compare definitions: v1 → v2

In [ ]:
result = compare_definitions(schema_v1, schema_v2)

print(f"Total diff issues : {len(result.issues)}")
print(f"is_valid          : {result.is_valid}")
print()

## 4. Display all issues

In [ ]:
for issue in sorted(result.issues, key=lambda i: (i.code, i.entity_id)):
    print(f"[{issue.severity.value.upper():4}] {issue.code:<30} {issue.entity_id}")
    print(f"       {issue.message}")
    if issue.context:
        print(f"       context: {issue.context}")
    print()

## 5. Summarise changes by code

In [ ]:
from collections import Counter


counts = Counter(i.code for i in result.issues)
print("Diff summary:")
for code, count in sorted(counts.items()):
    print(f"  {code:<35} x{count}")

## 6. Identical schemas → zero issues

In [ ]:
import copy

In [ ]:
schema_v1_copy = copy.deepcopy(schema_v1)
assert schema_v1_copy is not schema_v1

In [ ]:
self_result = compare_definitions(schema_v1, schema_v1)
print(f"Issues when comparing v1 to itself: {len(self_result.issues)}")
assert self_result.issues == [], "Expected zero issues for identical schemas"